# Sensitivity analysis cho `lambda_ctr` và `lambda_topo`

Notebook này chạy GeoODE-KD trên cùng teacher/student, corpus, seed, optimizer và evaluation; mỗi arm chỉ thay đổi hai hệ số loss cần khảo sát.

Thiết kế mặc định là **OFAT (one factor at a time)**:

- quét `lambda_ctr` với `lambda_topo = 0`;
- quét `lambda_topo` với `lambda_ctr = 0.5`;
- điểm `(lambda_ctr=0.5, lambda_topo=0)` được dùng làm reference chung và không chạy lặp.

Đặt `SWEEP_MODE = "factorial"` để chạy mọi tổ hợp và kiểm tra interaction. Mỗi cấu hình được lặp trên ba seed. Metric chính là `avg_all`; `avg_iod`, `avg_ood` là guardrail. Vì `lambda_topo` trực tiếp tối ưu topology, notebook còn đo out-of-objective trên một fixed probe: H0-W2 (thấp hơn tốt hơn), kNN@10 và Gram correlation (cao hơn tốt hơn).

> Sweep mặc định gồm 9 cấu hình x 3 seed = 27 lượt train. Hãy bật `SMOKE_TEST` để kiểm tra pipeline bằng 3 lượt train ngắn trước khi chạy đầy đủ.


In [ ]:
# 1. Experiment configuration
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
PAIR = "qwen3_0.6b_to_minilm_h384"
TEACHER_MODEL = "Qwen/Qwen3-Embedding-0.6B"
STUDENT_MODEL = "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base"
TEACHER_POOLING = "last_token"
STUDENT_POOLING = "cls"
TRAIN_DATA_REL = Path("data/train_set/train_100k.csv")
DATASET_TAG = "100k"

SWEEP_MODE = "ofat"  # "ofat" hoặc "factorial"
CTR_VALUES = [0.0, 0.25, 0.5, 0.75, 1.0]
TOPO_VALUES = [0.0, 0.25, 0.5, 0.75, 1.0]
CTR_REFERENCE = 0.5
TOPO_REFERENCE = 0.0
SEEDS = [42, 43, 44]

# Chạy nhanh để kiểm tra end-to-end: 1 seed, 3 cấu hình, 1 epoch.
SMOKE_TEST = False
RUN_TRAINING = True
INSTALL_REQUIREMENTS = False
STOP_ON_ERROR = True

HP = {
    "batch_size": 64,
    "epochs": 5,
    "learning_rate": 7e-5,
    "max_length": 256,
    "lambda_end": 1.0,
    "topo_metric": "chord",
}
NUM_WORKERS = 2
CUDA_VISIBLE_DEVICES = "0"
EVAL_RETRIEVAL = False
PAIR_THRESHOLD_SOURCE = "test"
EVAL_EVERY = 0

# Fixed-probe geometry. Teacher probe embedding được cache và tái sử dụng.
PROBE_SEED = 2026
PROBE_CORPUS_SENTENCES = 2048
PROBE_CORE_EVAL = 2048
PROBE_BATCH_SIZE = 256
H0_ROWS = 1000
KNN_ROWS = 2000
KNN_K = 10
GRAM_ROWS = 1500
GEOMETRY_SEED = 2026

SAVE_TO_GOOGLE_DRIVE = False
MAKE_LIGHT_ARCHIVE = True
# Điền tên run cũ để resume/phân tích lại đúng artifacts; để None cho run mới.
RUN_NAME_OVERRIDE = None
RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
RUN_NAME = RUN_NAME_OVERRIDE or f"lambda_sensitivity_{SWEEP_MODE}_{PAIR}_{DATASET_TAG}_{RUN_STAMP}"

assert SWEEP_MODE in {"ofat", "factorial"}
assert CTR_REFERENCE in CTR_VALUES and TOPO_REFERENCE in TOPO_VALUES
assert len(SEEDS) == len(set(SEEDS))
print(RUN_NAME)


In [ ]:
# 2. Locate project, optionally install dependencies, and validate runtime
import json
import os
import re
import shlex
import subprocess
import sys
import time

cwd = Path.cwd().resolve()
PROJECT_DIR = next(
    (p for p in (cwd, *cwd.parents) if (p / "main.py").is_file() and (p / "distiller.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

assert (PROJECT_DIR / ".git").exists(), f"Không phải Git clone: {PROJECT_DIR}"
assert (PROJECT_DIR / "main.py").is_file(), f"Không tìm thấy repo hợp lệ: {PROJECT_DIR}"
head_before = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
head_after = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Git HEAD: {head_before} -> {head_after}")

if INSTALL_REQUIREMENTS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
        check=True,
    )
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import numpy as np
import pandas as pd
import torch
from IPython.display import display

try:
    from google.colab import drive as colab_drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    colab_drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/embedding-kd-runs")
else:
    OUTPUT_BASE = PROJECT_DIR / "runs"

TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
RUN_ROOT = OUTPUT_BASE / RUN_NAME
ARMS_ROOT = RUN_ROOT / "arms"
CACHE_DIR = OUTPUT_BASE / "teacher_cache"
for path in (RUN_ROOT, ARMS_ROOT, CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)

assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
for split in ("train_set", "val_set", "test_set"):
    split_dir = PROJECT_DIR / "data" / split
    assert split_dir.is_dir() and any(split_dir.glob("*.csv")), f"Thiếu {split_dir}"
if not torch.cuda.is_available():
    raise RuntimeError("Sensitivity sweep cần GPU; hãy bật CUDA runtime.")
if hasattr(torch.cuda, "is_bf16_supported") and not torch.cuda.is_bf16_supported():
    raise RuntimeError("Qwen teacher dùng BF16; cần GPU Ampere hoặc mới hơn.")

print(f"Project: {PROJECT_DIR}")
print(f"Output:  {RUN_ROOT}")
for i in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}: {prop.name} ({prop.total_memory / 2**30:.1f} GiB)")


In [ ]:
# 3. Build the sweep plan and immutable commands
from itertools import product

def number_tag(value):
    return (f"{value:g}").replace("-", "m").replace(".", "p")

def sweep_points():
    if SWEEP_MODE == "factorial":
        points = list(product(CTR_VALUES, TOPO_VALUES))
    else:
        points = [(v, TOPO_REFERENCE) for v in CTR_VALUES]
        points += [(CTR_REFERENCE, v) for v in TOPO_VALUES if v != TOPO_REFERENCE]
    return list(dict.fromkeys(points))

points = sweep_points()
active_seeds = SEEDS
active_epochs = HP["epochs"]
if SMOKE_TEST:
    active_seeds = [SEEDS[0]]
    active_epochs = 1
    points = [
        (CTR_VALUES[0], TOPO_REFERENCE),
        (CTR_REFERENCE, TOPO_REFERENCE),
        (CTR_REFERENCE, TOPO_VALUES[1]),
    ]

ARM_PLAN = []
for seed in active_seeds:
    for lambda_ctr, lambda_topo in points:
        config_name = f"ctr_{number_tag(lambda_ctr)}__topo_{number_tag(lambda_topo)}"
        ARM_PLAN.append({
            "name": f"{config_name}__s{seed}",
            "config": config_name,
            "seed": seed,
            "lambda_ctr": float(lambda_ctr),
            "lambda_topo": float(lambda_topo),
        })

def build_command(arm):
    arm_dir = ARMS_ROOT / arm["name"]
    command = [
        sys.executable, str(PROJECT_DIR / "main.py"),
        "--method", "geoode",
        "--train_data", str(TRAIN_DATA),
        "--student_model", STUDENT_MODEL,
        "--teacher_model", TEACHER_MODEL,
        "--teacher_pooling", TEACHER_POOLING,
        "--student_pooling", STUDENT_POOLING,
        "--batch_size", str(HP["batch_size"]),
        "--epochs", str(active_epochs),
        "--save_every", str(active_epochs),
        "--lr", str(HP["learning_rate"]),
        "--max_length", str(HP["max_length"]),
        "--seed", str(arm["seed"]),
        "--save_dir", str(arm_dir),
        "--weights_dir", str(arm_dir / "weights"),
        "--cache_dir", str(CACHE_DIR),
        "--num_workers", str(NUM_WORKERS),
        "--pair_threshold_source", PAIR_THRESHOLD_SOURCE,
        "--eval_every", str(EVAL_EVERY),
        "--lambda_end", str(HP["lambda_end"]),
        "--lambda_ctr", str(arm["lambda_ctr"]),
        "--lambda_gram", "0",
        "--lambda_topo", str(arm["lambda_topo"]),
        "--topo_metric", HP["topo_metric"],
        "--projection_type", "pca",
        "--pca_center_fit",
        "--no-pca_subtract_mean",
        "--gauge_align",
        "--gauge_rotation", "procrustes",
        "--gauge_refit_every", "0",
        "--no_wandb",
    ]
    if not EVAL_RETRIEVAL:
        command.append("--no_eval_retrieval")
    return command

ARM_COMMANDS = {arm["name"]: build_command(arm) for arm in ARM_PLAN}

def scientific_signature(command):
    ignored = {"--save_dir", "--weights_dir", "--lambda_ctr", "--lambda_topo", "--seed"}
    signature, i = [], 0
    while i < len(command):
        if command[i] in ignored:
            i += 2
        else:
            signature.append(command[i])
            i += 1
    return signature

signatures = {tuple(scientific_signature(cmd)) for cmd in ARM_COMMANDS.values()}
assert len(signatures) == 1, "Các arm khác nhau ngoài seed/lambda/output path."
assert len({arm["name"] for arm in ARM_PLAN}) == len(ARM_PLAN)

run_config = {
    "run_name": RUN_NAME, "sweep_mode": SWEEP_MODE, "smoke_test": SMOKE_TEST,
    "teacher_model": TEACHER_MODEL, "student_model": STUDENT_MODEL,
    "teacher_pooling": TEACHER_POOLING, "student_pooling": STUDENT_POOLING,
    "train_data": str(TRAIN_DATA), "seeds": active_seeds, "points": points,
    "ctr_values": CTR_VALUES, "topo_values": TOPO_VALUES,
    "ctr_reference": CTR_REFERENCE, "topo_reference": TOPO_REFERENCE,
    "hp": {**HP, "epochs": active_epochs}, "arms": ARM_PLAN,
}
config_path = RUN_ROOT / "run_config.json"
if config_path.is_file():
    existing = json.loads(config_path.read_text(encoding="utf-8"))
    assert existing == json.loads(json.dumps(run_config)), "RUN_NAME đã có config khác."
else:
    config_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")

plan = pd.DataFrame(ARM_PLAN)
print(f"Sweep: {len(points)} configs x {len(active_seeds)} seeds = {len(ARM_PLAN)} runs")
display(plan)
print("Ví dụ command:")
print(shlex.join(ARM_COMMANDS[ARM_PLAN[0]["name"]]))


In [ ]:
# 4. Train or resume completed arms. A partial arm is never silently reused.
PROGRESS_EVERY_SEC = 30

def read_records(path):
    if not path.is_file():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def has_final_test(path):
    return any(record.get("test") and not record.get("train") for record in read_records(path))

def stream_process(command, log_path, env):
    last_progress = 0.0
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            log_handle.write(line)
            if "%|" not in line:
                print(line.rstrip())
            elif time.monotonic() - last_progress >= PROGRESS_EVERY_SEC:
                print(line.rstrip()[-180:])
                last_progress = time.monotonic()
        return process.wait()

env = os.environ.copy()
env.update({
    "CUDA_VISIBLE_DEVICES": CUDA_VISIBLE_DEVICES,
    "TOKENIZERS_PARALLELISM": "false",
    "WANDB_MODE": "disabled",
    "TQDM_MININTERVAL": str(PROGRESS_EVERY_SEC),
})
run_status = []

if not RUN_TRAINING:
    print("RUN_TRAINING=False: chỉ tạo plan, không train.")
else:
    for position, arm in enumerate(ARM_PLAN, start=1):
        arm_dir = ARMS_ROOT / arm["name"]
        metrics_path = arm_dir / "metrics.jsonl"
        log_path = ARMS_ROOT / f"{arm['name']}_train.log"
        if has_final_test(metrics_path):
            print(f"[SKIP {position}/{len(ARM_PLAN)}] {arm['name']} đã hoàn tất")
            run_status.append({"arm": arm["name"], "status": "skipped_complete", "seconds": 0.0})
            continue
        if metrics_path.exists():
            raise RuntimeError(f"Arm dở dang: {arm_dir}. Xóa riêng arm này hoặc dùng RUN_NAME mới.")
        arm_dir.mkdir(parents=True, exist_ok=True)
        print("\n" + "#" * 88)
        print(
            f"RUN {position}/{len(ARM_PLAN)}: {arm['name']} | "
            f"ctr={arm['lambda_ctr']}, topo={arm['lambda_topo']}"
        )
        started = time.perf_counter()
        return_code = stream_process(ARM_COMMANDS[arm["name"]], log_path, env)
        elapsed = time.perf_counter() - started
        status = "complete" if return_code == 0 and has_final_test(metrics_path) else "failed"
        run_status.append({"arm": arm["name"], "status": status, "seconds": elapsed})
        print(f"[{status.upper()}] {elapsed / 60:.1f} min | log={log_path}")
        if status == "failed" and STOP_ON_ERROR:
            raise RuntimeError(f"{arm['name']} failed; xem {log_path}")

status_df = pd.DataFrame(run_status)
status_df.to_csv(RUN_ROOT / "run_status.csv", index=False)
display(status_df)


In [ ]:
# 5. Build one fixed probe, cache teacher embeddings, and encode every final student
import transformers
from transformers import AutoModel, AutoTokenizer

from src.probe_set import build_probe_set, probe_digest
from src.structural_audit import encode_texts, load_student

PROBE_DIR = OUTPUT_BASE / "probe" / PAIR
PROBE_DIR.mkdir(parents=True, exist_ok=True)
PROBE_PATH = PROBE_DIR / f"probe_{DATASET_TAG}_seed{PROBE_SEED}_noretrieval.csv"
if PROBE_PATH.is_file():
    PROBE = pd.read_csv(PROBE_PATH, keep_default_na=False)
else:
    PROBE = build_probe_set(
        PROJECT_DIR, TRAIN_DATA, n_corpus=PROBE_CORPUS_SENTENCES,
        n_docs_per_retrieval=0, eval_splits=("test_set",),
        core_eval=PROBE_CORE_EVAL, core_retrieval=0, seed=PROBE_SEED,
    )
    PROBE = PROBE[~PROBE["group"].str.startswith("retrieval")].reset_index(drop=True)
    PROBE["probe_id"] = range(len(PROBE))
    PROBE.to_csv(PROBE_PATH, index=False)

PROBE_DIGEST = probe_digest(PROBE)
PROBE_TEXTS = PROBE["text"].astype(str).tolist()
DEVICE = torch.device("cuda")

def slug(name):
    return re.sub(r"[^A-Za-z0-9]+", "-", name).strip("-").lower()

PROBE_TEACHER_PATH = PROBE_DIR / (
    f"teacher_{slug(TEACHER_MODEL)}_{TEACHER_POOLING}_{PROBE_DIGEST}.pt"
)
if PROBE_TEACHER_PATH.is_file():
    teacher_payload = torch.load(PROBE_TEACHER_PATH, map_location="cpu", weights_only=False)
    assert teacher_payload["probe_digest"] == PROBE_DIGEST
    print(f"[teacher probe] reuse {PROBE_TEACHER_PATH.name}")
else:
    dtype_kw = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"
    teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL, trust_remote_code=True)
    teacher_model = AutoModel.from_pretrained(
        TEACHER_MODEL, trust_remote_code=True, **{dtype_kw: torch.bfloat16}
    ).to(DEVICE).eval()
    teacher_out = encode_texts(
        teacher_model, teacher_tokenizer, PROBE_TEXTS, device=DEVICE,
        pooling=TEACHER_POOLING, batch_size=64, max_length=HP["max_length"],
        layers=False, amp=False, progress=True,
    )
    teacher_payload = {
        "embeddings": teacher_out["final"].float(),
        "probe_digest": PROBE_DIGEST, "teacher_model_name": TEACHER_MODEL,
        "pooling_method": TEACHER_POOLING, "max_length": HP["max_length"],
    }
    torch.save(teacher_payload, PROBE_TEACHER_PATH)
    del teacher_model, teacher_out
    torch.cuda.empty_cache()

student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
for position, arm in enumerate(ARM_PLAN, start=1):
    arm_dir = ARMS_ROOT / arm["name"]
    probe_final_path = arm_dir / "probe_final.pt"
    if probe_final_path.is_file():
        payload = torch.load(probe_final_path, map_location="cpu", weights_only=False)
        assert payload["probe_digest"] == PROBE_DIGEST
        continue
    weights_path = arm_dir / "weights" / f"student_epoch_{active_epochs}.pt"
    if not weights_path.is_file():
        raise FileNotFoundError(f"Thiếu final weights: {weights_path}")
    print(f"[student probe {position}/{len(ARM_PLAN)}] {arm['name']}")
    student_model = load_student(STUDENT_MODEL, weights_path, DEVICE)
    student_out = encode_texts(
        student_model, student_tokenizer, PROBE_TEXTS, device=DEVICE,
        pooling=STUDENT_POOLING, batch_size=PROBE_BATCH_SIZE,
        max_length=HP["max_length"], layers=False, progress=True,
    )
    torch.save({
        "probe_digest": PROBE_DIGEST, "arm": arm["name"],
        "seed": arm["seed"], "lambda_ctr": arm["lambda_ctr"],
        "lambda_topo": arm["lambda_topo"],
        "final": student_out["final"].to(torch.float16),
    }, probe_final_path)
    del student_model, student_out
    torch.cuda.empty_cache()

print(f"Probe: {len(PROBE)} rows, digest={PROBE_DIGEST}")


In [ ]:
# 6. Collect downstream metrics and fixed-probe geometry
import torch.nn.functional as F
from scipy.sparse.csgraph import minimum_spanning_tree

from src import structural_audit as sa

def h0_death_times(x, metric="chord"):
    x = F.normalize(torch.as_tensor(x).float(), dim=-1)
    similarity = (x @ x.T).clamp(-1.0, 1.0).numpy()
    if metric == "chord":
        distance = np.sqrt(np.maximum(2.0 - 2.0 * similarity, 1e-7))
    elif metric == "angular":
        distance = np.arccos(np.clip(similarity, -1.0 + 1e-7, 1.0 - 1e-7))
    elif metric == "cosine":
        distance = np.maximum(1.0 - similarity, 0.0)
    else:
        raise ValueError(metric)
    distance = np.maximum(distance.astype(np.float64), 1e-12)
    np.fill_diagonal(distance, 0.0)
    deaths = np.sort(minimum_spanning_tree(distance).data)
    if len(deaths) != len(distance) - 1:
        raise RuntimeError(f"MST có {len(deaths)} edges cho {len(distance)} points")
    return deaths

teacher = teacher_payload["embeddings"].float()
n_probe = len(teacher)
assert n_probe > KNN_K
rng = np.random.default_rng(GEOMETRY_SEED)
h0_idx = np.sort(rng.choice(n_probe, size=min(H0_ROWS, n_probe), replace=False))
knn_idx = np.sort(rng.choice(n_probe, size=min(KNN_ROWS, n_probe), replace=False))
gram_idx = np.sort(rng.choice(n_probe, size=min(GRAM_ROWS, n_probe), replace=False))
teacher_h0 = h0_death_times(teacher[h0_idx], HP["topo_metric"])
teacher_knn = sa.knn_indices(teacher[knn_idx], KNN_K)

summary_keys = ("avg_iod", "avg_ood", "avg_retrieval", "avg_all")
rows = []
for position, arm in enumerate(ARM_PLAN, start=1):
    records = read_records(ARMS_ROOT / arm["name"] / "metrics.jsonl")
    final_test = next((r["test"] for r in records if r.get("test") and not r.get("train")), None)
    train_epochs = [r["train"] for r in records if r.get("train")]
    if final_test is None:
        raise RuntimeError(f"Thiếu final test: {arm['name']}")
    summary = final_test["summary"]
    last_train = train_epochs[-1] if train_epochs else {}
    payload = torch.load(
        ARMS_ROOT / arm["name"] / "probe_final.pt", map_location="cpu", weights_only=False
    )
    assert payload["probe_digest"] == PROBE_DIGEST
    student = payload["final"].float()
    student_h0 = h0_death_times(student[h0_idx], HP["topo_metric"])
    student_knn = sa.knn_indices(student[knn_idx], KNN_K)
    rows.append({
        **arm,
        **{key: (np.nan if summary.get(key) is None else 100.0 * float(summary[key])) for key in summary_keys},
        "train_loss_total": last_train.get("loss_total", last_train.get("loss")),
        "train_loss_end": last_train.get("loss_end"),
        "train_loss_ctr": last_train.get("loss_ctr"),
        "train_loss_topo": last_train.get("loss_topo"),
        "h0_w2": float(np.sqrt(np.mean((student_h0 - teacher_h0) ** 2))),
        "knn_at_10": sa.knn_overlap(
            student[knn_idx], teacher[knn_idx], KNN_K,
            neighbours_a=student_knn, neighbours_b=teacher_knn,
        ),
        "rho_gram": sa.gram_correlation(
            student[gram_idx], teacher[gram_idx], max_rows=len(gram_idx), seed=GEOMETRY_SEED
        ),
    })
    print(f"[metrics {position}/{len(ARM_PLAN)}] {arm['name']}")

results = pd.DataFrame(rows).sort_values(["lambda_ctr", "lambda_topo", "seed"]).reset_index(drop=True)
assert len(results) == len(ARM_PLAN)
results.to_csv(RUN_ROOT / "metrics_by_run.csv", index=False)
display(results.style.format(precision=5, na_rep="-"))


In [ ]:
# 7. Aggregate seeds, calculate paired deltas from the common reference, and rank configs
METRICS = ["avg_all", "avg_iod", "avg_ood", "h0_w2", "knn_at_10", "rho_gram"]
DIRECTION = {"avg_all": 1, "avg_iod": 1, "avg_ood": 1, "h0_w2": -1, "knn_at_10": 1, "rho_gram": 1}

aggregate = results.groupby(["lambda_ctr", "lambda_topo"], as_index=False).agg(
    n_seeds=("seed", "nunique"),
    **{f"{metric}_mean": (metric, "mean") for metric in METRICS},
    **{f"{metric}_std": (metric, "std") for metric in METRICS},
)
aggregate.to_csv(RUN_ROOT / "aggregate_by_config.csv", index=False)

reference = results[
    np.isclose(results.lambda_ctr, CTR_REFERENCE) & np.isclose(results.lambda_topo, TOPO_REFERENCE)
][["seed", *METRICS]].copy()
assert len(reference) == len(active_seeds), "Thiếu reference cho một hoặc nhiều seed."
reference = reference.rename(columns={metric: f"{metric}_reference" for metric in METRICS})
paired = results.merge(reference, on="seed", validate="many_to_one")
for metric in METRICS:
    paired[f"delta_{metric}"] = paired[metric] - paired[f"{metric}_reference"]
paired.to_csv(RUN_ROOT / "paired_deltas_from_reference.csv", index=False)

def axis_summary(frame, parameter):
    delta_columns = [f"delta_{metric}" for metric in METRICS]
    named = {f"{column}_mean": (column, "mean") for column in delta_columns}
    named.update({f"{column}_std": (column, "std") for column in delta_columns})
    for metric in METRICS:
        desired = DIRECTION[metric]
        named[f"{metric}_desired_signs"] = (
            f"delta_{metric}", lambda values, desired=desired: int((desired * values > 0).sum())
        )
    return frame.groupby(parameter, as_index=False).agg(**named)

ctr_slice = paired[np.isclose(paired.lambda_topo, TOPO_REFERENCE)].copy()
topo_slice = paired[np.isclose(paired.lambda_ctr, CTR_REFERENCE)].copy()
ctr_sensitivity = axis_summary(ctr_slice, "lambda_ctr")
topo_sensitivity = axis_summary(topo_slice, "lambda_topo")
ctr_sensitivity.to_csv(RUN_ROOT / "ctr_sensitivity.csv", index=False)
topo_sensitivity.to_csv(RUN_ROOT / "topo_sensitivity.csv", index=False)

best = aggregate.sort_values("avg_all_mean", ascending=False).iloc[0]
print(
    f"Best observed avg_all: ctr={best.lambda_ctr:g}, topo={best.lambda_topo:g}, "
    f"mean={best.avg_all_mean:.4f}, std={best.avg_all_std:.4f}"
)
print("\nAggregate by config")
display(aggregate.style.format(precision=5, na_rep="-"))
print("\nPaired sensitivity: lambda_ctr (delta vs reference within seed)")
display(ctr_sensitivity.style.format(precision=5, na_rep="-"))
print("\nPaired sensitivity: lambda_topo (delta vs reference within seed)")
display(topo_sensitivity.style.format(precision=5, na_rep="-"))
print("\nLưu ý: đây là observed sensitivity trên grid đã chọn; không suy causal/significance từ 3 seeds.")


In [ ]:
# 8. Sensitivity curves; in factorial mode also draw response heatmaps
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plot_metrics = [
    ("avg_all", "Downstream avg_all (x100)", "higher is better"),
    ("h0_w2", "H0-W2 on fixed probe", "lower is better"),
    ("knn_at_10", "kNN@10 preservation", "higher is better"),
]
fig, axes = plt.subplots(2, len(plot_metrics), figsize=(14, 7.5))
for row_idx, (frame, parameter, fixed_label) in enumerate([
    (ctr_slice, "lambda_ctr", f"lambda_topo={TOPO_REFERENCE:g}"),
    (topo_slice, "lambda_topo", f"lambda_ctr={CTR_REFERENCE:g}"),
]):
    x_values = sorted(frame[parameter].unique())
    positions = np.arange(len(x_values))
    for ax, (metric, title, direction) in zip(axes[row_idx], plot_metrics):
        grouped = frame.groupby(parameter)[metric].agg(["mean", "std"]).reindex(x_values)
        yerr = grouped["std"].fillna(0.0)
        ax.errorbar(positions, grouped["mean"], yerr=yerr, marker="o", capsize=4)
        for seed in active_seeds:
            seed_rows = frame[frame.seed == seed].set_index(parameter).reindex(x_values)
            ax.plot(positions, seed_rows[metric], alpha=0.25, linewidth=1)
        ax.set_xticks(positions, [f"{value:g}" for value in x_values])
        ax.set_xlabel(parameter)
        ax.set_title(f"{title}\n{fixed_label}; {direction}")
fig.suptitle("Paired sensitivity curves: mean +/- std; faint lines are individual seeds")
fig.tight_layout()
curve_path = RUN_ROOT / "sensitivity_curves.png"
fig.savefig(curve_path, dpi=180, bbox_inches="tight")
plt.show()

if SWEEP_MODE == "factorial" and not SMOKE_TEST:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, (metric, title, _) in zip(axes, plot_metrics):
        pivot = aggregate.pivot(index="lambda_topo", columns="lambda_ctr", values=f"{metric}_mean")
        image = ax.imshow(pivot.values, aspect="auto", origin="lower", cmap="viridis")
        ax.set_xticks(range(len(pivot.columns)), [f"{v:g}" for v in pivot.columns])
        ax.set_yticks(range(len(pivot.index)), [f"{v:g}" for v in pivot.index])
        ax.set(xlabel="lambda_ctr", ylabel="lambda_topo", title=title)
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                ax.text(j, i, f"{pivot.iloc[i, j]:.3f}", ha="center", va="center", fontsize=8)
        fig.colorbar(image, ax=ax, shrink=0.8)
    fig.suptitle("Full-factorial response surfaces (mean across seeds)")
    fig.tight_layout()
    heatmap_path = RUN_ROOT / "factorial_response_surfaces.png"
    fig.savefig(heatmap_path, dpi=180, bbox_inches="tight")
    plt.show()


In [ ]:
# 9. Export lightweight analysis artifacts (no checkpoints or probe tensors)
import zipfile

archive_path = RUN_ROOT.parent / f"{RUN_NAME}_summary.zip"
if MAKE_LIGHT_ARCHIVE:
    include_suffixes = {".csv", ".json", ".png", ".log"}
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(RUN_ROOT.rglob("*")):
            if path.is_file() and path.suffix.lower() in include_suffixes:
                archive.write(path, path.relative_to(RUN_ROOT))
    print(f"Created {archive_path} ({archive_path.stat().st_size / 2**20:.1f} MiB)")
    if IN_COLAB and not SAVE_TO_GOOGLE_DRIVE:
        from google.colab import files
        files.download(str(archive_path))
